# Query expansion with a language model
flan-t5-large writes a passage that answers each query, and the passage is appended to the query (Query2doc, Wang et al. 2023). The original query is repeated five times so that it keeps its weight. Zero-shot and few-shot prompts are compared on the test queries.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import json
import time

import torch
from transformers import T5ForConditionalGeneration, T5Tokenizer
import pandas as pd
import pyterrier as pt
from pyterrier.measures import nDCG

from helpers import (INDEX_FOLDER, RANKING_METRICS, build_retriever, clean_query_text, controls_for_model,
                     load_best_parameters, load_documents, load_queries_and_qrels, milliseconds_per_query,
                     open_index, save_results)

if not pt.java.started():
    pt.java.init()

/opt/anaconda3/envs/searchengines/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1018 23:39:56.926000 8483 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [2]:
documents = load_documents()
train_queries, train_qrels = load_queries_and_qrels("train")
test_queries, test_qrels = load_queries_and_qrels("test")

best_parameters = load_best_parameters()
index = open_index("stopwords_stemming")

bm25 = build_retriever(index, "BM25", controls_for_model("BM25", best_parameters["BM25"]))
dirichlet = build_retriever(index, "DirichletLM", controls_for_model("DirichletLM", best_parameters["DirichletLM"]))

[INFO] Opening /Users/carlson/.ir_datasets/beir/fiqa/docs.pklz4/bin with direct file access


## Model

In [3]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-large")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-large").to(device)

Loading weights: 100%|██████████████████████| 558/558 [00:00<00:00, 6136.25it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
BATCH_SIZE = 8
MAX_NEW_TOKENS = 50
QUERY_REPETITIONS = 5
NUMBER_OF_EXEMPLARS = 2


def generate_passages(prompts: list) -> list:
    passages = []
    for batch_start in range(0, len(prompts), BATCH_SIZE):
        batch_prompts = prompts[batch_start:batch_start + BATCH_SIZE]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True, truncation=True).to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        passages.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))

        if device.type == "mps":
            torch.mps.empty_cache()
    return passages


def generate_passages_cached(cache_name: str, prompts: list) -> dict:
    cache_path = f"{INDEX_FOLDER}/{cache_name}.json"
    if os.path.exists(cache_path):
        with open(cache_path) as cache_file:
            return json.load(cache_file)

    start_time = time.time()
    passages = generate_passages(prompts)
    seconds_per_query = (time.time() - start_time) / len(prompts)

    generated = {"seconds_per_query": seconds_per_query, "passages": passages}
    with open(cache_path, "w") as cache_file:
        json.dump(generated, cache_file)
    return generated


def append_passages_to_queries(queries: pd.DataFrame, passages: list) -> pd.DataFrame:
    expanded_queries = queries.copy()
    expanded_queries["query"] = [
        " ".join([query] * QUERY_REPETITIONS) + " " + clean_query_text(passage)
        for query, passage in zip(queries["query"], passages)
    ]
    return expanded_queries

## Prompts
The few-shot prompt shows two relevant passages from the train queries. Only the first 100 characters of each passage are used.

In [5]:
exemplar_qrels = train_qrels[train_qrels["label"] > 0].sample(n=NUMBER_OF_EXEMPLARS, random_state=777)
exemplars = exemplar_qrels.merge(documents, on="docno").merge(train_queries, on="qid")

zero_shot_prompts = [f"Write a passage that answers the following query: {query}\nPassage:"
                     for query in test_queries["query"]]

few_shot_prefix = "Write a passage that answers the given query:\n"
for exemplar in exemplars.itertuples():
    few_shot_prefix += f"Query: {exemplar.query}\nPassage: {exemplar.text[:100]}\n"
few_shot_prompts = [f"{few_shot_prefix}Query: {query}\nPassage:" for query in test_queries["query"]]

In [6]:
zero_shot_generated = generate_passages_cached("flan_t5_zero_shot_test", zero_shot_prompts)
few_shot_generated = generate_passages_cached("flan_t5_few_shot_test", few_shot_prompts)

zero_shot_queries = append_passages_to_queries(test_queries, zero_shot_generated["passages"])
few_shot_queries = append_passages_to_queries(test_queries, few_shot_generated["passages"])

zero_shot_queries["query"].iloc[0][:300]

'Where should I park my rainy day emergency fund Where should I park my rainy day emergency fund Where should I park my rainy day emergency fund Where should I park my rainy day emergency fund Where should I park my rainy day emergency fund The emergency fund is a savings account that is used to pay '

## Test results
Query time includes the time flan-t5 spends writing the passage.

In [7]:
query_variants = {
    "no expansion": (test_queries, 0.0),
    "zero-shot": (zero_shot_queries, zero_shot_generated["seconds_per_query"]),
    "few-shot": (few_shot_queries, few_shot_generated["seconds_per_query"]),
}

result_tables = []
for variant_name, (variant_queries, generation_seconds_per_query) in query_variants.items():
    for model_name, retriever in [("BM25", bm25), ("DirichletLM", dirichlet)]:
        system_name = f"{model_name} ({variant_name})"
        result = pt.Experiment([retriever], variant_queries, test_qrels, RANKING_METRICS,
                               names=[system_name], round=4)

        retrieval_milliseconds = milliseconds_per_query(retriever, variant_queries)
        result["query time (ms)"] = round(retrieval_milliseconds + 1000 * generation_seconds_per_query, 1)
        result_tables.append(result)

test_results = pd.concat(result_tables, ignore_index=True)
test_results

,name,P@10,R@100,nDCG@5,nDCG@10,nDCG@20,RR@10,query time (ms)
0,BM25 (no expansion),0.0702,0.5524,0.2309,0.2541,0.2788,0.3119,14.9
1,DirichletLM (no expansion),0.0685,0.5412,0.2388,0.2549,0.2774,0.3199,15.9
2,BM25 (zero-shot),0.0707,0.5592,0.2314,0.2554,0.2836,0.3161,439.5
3,DirichletLM (zero-shot),0.0525,0.4736,0.1725,0.1908,0.2097,0.2363,442.1
4,BM25 (few-shot),0.0707,0.5594,0.2364,0.2554,0.2825,0.3145,413.2
5,DirichletLM (few-shot),0.0562,0.4799,0.1734,0.1946,0.2160,0.2354,415.7


In [8]:
save_results(test_results, "03_llm_query_expansion")

## Which queries are helped
nDCG@10 per test query for BM25, BM25 with RM3 and BM25 with zero-shot expansion.

In [9]:
rm3_settings = best_parameters["RM3"]
rm3_bm25 = bm25 >> pt.rewrite.RM3(index, fb_terms=rm3_settings["fb_terms"], fb_docs=rm3_settings["fb_docs"]) >> bm25


def ndcg_per_query(retriever, queries: pd.DataFrame, column_name: str) -> pd.Series:
    results = pt.Experiment([retriever], queries, test_qrels, [nDCG @ 10], perquery=True)
    return results.set_index("qid")["value"].rename(column_name)


per_query_scores = pd.concat([
    ndcg_per_query(bm25, test_queries, "BM25"),
    ndcg_per_query(rm3_bm25, test_queries, "RM3"),
    ndcg_per_query(bm25, zero_shot_queries, "zero-shot"),
], axis=1)
per_query_scores["query"] = test_queries.set_index("qid")["query"]

expansion_columns = ["RM3", "zero-shot"]
pd.DataFrame({
    "better than BM25": per_query_scores[expansion_columns].gt(per_query_scores["BM25"], axis=0).sum(),
    "worse than BM25": per_query_scores[expansion_columns].lt(per_query_scores["BM25"], axis=0).sum(),
})

,better than BM25,worse than BM25
RM3,97,94
zero-shot,80,80


In [10]:
pd.set_option("display.max_colwidth", None)

score_columns = ["BM25", "RM3", "zero-shot"]
queries_without_any_hit = per_query_scores[(per_query_scores[score_columns] == 0).all(axis=1)]

print(f"{len(queries_without_any_hit)} of {len(per_query_scores)} queries have nDCG@10 of 0 for all three")
queries_without_any_hit.head(5)

300 of 648 queries have nDCG@10 of 0 for all three


,BM25,RM3,zero-shot,query
qid,,,,
10039,0.0,0.0,0.0,Do individual investors use Google to obtain stock quotes
10136,0.0,0.0,0.0,How to minimise the risk of a reduction in purchase power in case of Brexit for money held in a bank account
10137,0.0,0.0,0.0,F 1 student investing in foreign markets
10183,0.0,0.0,0.0,How are various types of income taxed differently in the USA
10246,0.0,0.0,0.0,Understanding the T 3 settlement days rule
